In [1]:
from playwright.async_api import async_playwright
import asyncio
import nest_asyncio
import pandas as pd
import cv2
import io
import numpy as np
import easyocr
from datetime import datetime
import os
import matplotlib.pyplot as plt
import json
from urllib.parse import urljoin

In [2]:
court = "Orissa"

In [3]:
reader = easyocr.Reader(['en'], gpu=True)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [4]:
bail_data = pd.read_csv(rf'Bail - Case dataset/Bail {court}.csv')
cnr_numbers = bail_data['CNR_NUMBER'].tolist()

/tmp/ipykernel_270686/2107866343.py:1: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  bail_data = pd.read_csv(rf'Bail - Case dataset/Bail {court}.csv')


In [6]:
# cnr_numbers.index('GAHC030000422021')

In [7]:
dataframe = pd.read_csv(rf'Bail - Case dataset/Bail {court}.csv')
pd_row = dataframe.iloc[0]
print(pd_row)

CNR_NUMBER                          ODHC010384572015
CASE_NUMBER                                    19196
CASE_TYPE                                     ABLAPL
CASETYPE_FULLFORM                  ANTICIPATORY BAIL
CIVIL_CRIMINAL                              CRIMINAL
SUB_CLASSIFICATION                               NaN
COMBINED_CASE_NUMBER               ABLAPL-19196-2015
COURT_NAME                    Principal Bench Orissa
COURT_NUMBER                                    1549
NAME_OF_HIGH_COURT                 ORISSA HIGH COURT
CURRENT_STAGE                             FOR ORDERS
CURRENT_STATUS                               Pending
DATE_FILED                                15-12-2015
DECISION_DATE                                    NaN
FILING_NUMBER                                  19196
HEARING_COUNT                                      7
LAST_SYNC_TIME                            17-02-2021
NATURE_OF_DISPOSAL                               NaN
NATURE_OF_DISPOSAL_OUTCOME                    

/tmp/ipykernel_270686/3070148826.py:1: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(rf'Bail - Case dataset/Bail {court}.csv')


In [8]:
# cnr_numbers = cnr_numbers[37895:]

In [9]:
cnr_numbers[:15]

['ODHC010384572015',
 'ODHC010321972015',
 'ODHC010408032015',
 'ODHC010537122015',
 'ODHC010142742015',
 'ODHC010031462015',
 'ODHC010066572015',
 'ODHC010122542015',
 'ODHC010631292015',
 'ODHC010255402015',
 'ODHC010142752015',
 'ODHC010315012015',
 'ODHC010241142015',
 'ODHC010701342015',
 'ODHC010325262015']

In [10]:
# UPHC012007592012
test_cnr = [
    'UPHC010933632010',
    'UPHC011683092020',
    'UPHC010879282018',
    'UPHC011942132010',
    'UPHC012007592012',
    'UPHC010863932010',
    'UPHC011683072016',
    'UPHC011705422020',
    'UPHC011915772010',
    'UPHC011993542011'
    ]

In [11]:
test_cnr2 = [
    'UPHC011683092020',
    'UPHC010879282018',
    'UPHC012007592012',
    'UPHC011683072016',
    'UPHC011705422020',
]

In [12]:
async def ML_captcha_solver3(image_path):
    img = cv2.imread(image_path)
    text = reader.readtext(img, detail=0)
    if len(text) == 0:
        return ''
    return text[0].strip()

In [13]:
async def ML_captcha_solver4(img):
    text = reader.readtext(img, detail=0)
    if len(text) == 0:
        return '000000'
    captcha_text = text[0].strip()[:6] if len(text) > 0 else '000000'
    captcha_text = ''.join(['0' if not ch.isalnum() else ch for ch in captcha_text])
    if len(captcha_text) < 6:
        captcha_text = captcha_text.ljust(6, '0')
    # print(captcha_text)
    return captcha_text

In [14]:
async def save_captcha(page):
    await page.screenshot(path='captcha.png', clip={"x": 522, "y": 355, "width": 113, "height": 42})

In [15]:
async def get_captcha(page):
    screenshot = await page.screenshot(clip={"x": 522, "y": 355, "width": 113, "height": 42})
    img = cv2.imdecode(np.frombuffer(screenshot, np.uint8), cv2.IMREAD_COLOR)
    # plt.imshow(img)
    return img

In [ ]:
# (521,353),(640, 398)

: 

: 

In [16]:
case_info = []

In [17]:
from playwright.async_api import TimeoutError

async def get_links(cnr_number, page):
    links = await page.query_selector_all('a[href*="cases/display_pdf.php?"]')
    base_url = 'https://hcservices.ecourts.gov.in/hcservices/main.php'
    print(f'CNR: {cnr_number}')
    link_list = []
    if len(links) == 0:
        return
    for link in links:
        href = await link.get_attribute('href')
        print(f'Link: {href}')
        if href:
            full_url = urljoin(base_url, href)
            print(full_url)
            link_list.append(full_url)
    nature_of_disposal = ""
    try:
        locator = page.locator('//tr[td[1][contains(normalize-space(), "Nature of Disposal")]]/td[2]')
        nature_of_disposal = await locator.text_content(timeout=10000)
    except TimeoutError as e:
        print(f'Timeout error: {e}')
        return
    except Exception as e:
        print(f'Error: {e}')
        
    elem = {
        'cnr_number': cnr_number,
        'links': link_list,
        'result': nature_of_disposal
    }
    case_info.append(elem)

In [18]:
import os
output_dir = f"PDF_{court}"
os.makedirs(output_dir, exist_ok=True)

async def get_links(cnr_number, page, context):
    links = await page.query_selector_all('a[href*="cases/display_pdf.php?"]')
    base_url = 'https://hcservices.ecourts.gov.in/hcservices/main.php'
    print(f'CNR: {cnr_number}')
    link_list = []
    if len(links) == 0:
        return
    for link in links:
        href = await link.get_attribute('href')
        print(f'Link: {href}')
        if href:
            full_url = urljoin(base_url, href)
            print(full_url)
            link_list.append(full_url)

    if not link_list:
        return

    # Download only the last link
    last_link = link_list[-1]
    try:
        response = await context.request.get(last_link)
        if response.ok:
            file_path = os.path.join(output_dir, f"{cnr_number}.pdf")
            with open(file_path, "wb") as f:
                f.write(await response.body())
            print(f"Saved PDF for {cnr_number}")
        else:
            print(f"Failed to download PDF for {cnr_number} - status {response.status}")
    except Exception as e:
        print(f"Download error for {cnr_number}: {e}")


In [19]:
async def check_order_uploaded(page):
    try:
        task_orders = asyncio.create_task(
            page.wait_for_selector('h2.h2class', timeout=10000)  # Orders present
        )
        task_not_uploaded = asyncio.create_task(
            page.wait_for_selector('p.blinking', timeout=10000)  # Not uploaded
        )

        done, pending = await asyncio.wait(
            [task_orders, task_not_uploaded],
            return_when=asyncio.FIRST_COMPLETED
        )

        # Cancel remaining tasks
        for task in pending:
            task.cancel()

        # Get the completed task
        done_task = list(done)[0]
        selector = await done_task

        # Check which element was matched
        tag_name = await selector.evaluate("el => el.tagName")
        return tag_name == 'H2'
    except Exception:
        return False

In [ ]:
# import asyncio

# async def is_captcha_valid(page, timeout=10000):
#     # Wait until #caseHistoryDiv is visible (not display: none)
#     # await page.wait_for_function(
#     #     """() => {
#     #         const div = document.querySelector('#caseHistoryDiv');
#     #         return div && window.getComputedStyle(div).display !== 'none';
#     #     }""",
#     #     timeout=timeout,
#     #     polling=250
#     # )

#     # Create tasks for parallel waiting
#     task_valid = asyncio.create_task(
#         page.wait_for_function(
#             """() => {
#                 const div = document.querySelector('#caseHistoryDiv');
#                 return div && div.innerHTML.includes('<title>Case History</title>');
#             }""",
#             timeout=timeout,
#             polling=250
#         )
#     )
#     task_invalid_cnr = asyncio.create_task(
#         page.wait_for_function(
#             """() => {
#                 const div = document.querySelector('#caseHistoryDiv');
#                 return div && div.innerText.toLowerCase().includes('cnr does not exits');
#             }
#             """
#         )
#     )
#     task_invalid_p = asyncio.create_task(
#         page.wait_for_selector('div#caseHistoryDiv p:has-text("Invalid Captcha")', timeout=timeout)
#     )
#     task_invalid_text = asyncio.create_task(
#         page.wait_for_function(
#             """() => {
#                 const div = document.querySelector('#caseHistoryDiv');
#                 return div && div.innerText.toLowerCase().includes('invalid captcha');
#             }""",
#             timeout=timeout,
#             polling=250
#         )
#     )

#     done, pending = await asyncio.wait(
#         [task_valid, task_invalid_cnr, task_invalid_p, task_invalid_text],
#         return_when=asyncio.FIRST_COMPLETED
#     )

#     # Cancel unused tasks
#     for task in pending:
#         task.cancel()

#     # If the successful task is the one checking for <title>Case History</title>
#     if task_invalid_p not in done and task_invalid_text not in done:
#         return True
#     else:
#         return False


In [24]:
#naya wala
async def is_captcha_valid(page, timeout=10000):
    async def wait_valid():
        try:
            await page.wait_for_function(
                """() => {
                    const div = document.querySelector('#caseHistoryDiv');
                    return div && div.innerHTML.includes('<title>Case History</title>');
                }""",
                timeout=timeout,
                polling=250
            )
            return 'valid'
        except Exception:
            return 'timeout_valid'

    async def wait_invalid_texts():
        try:
            await page.wait_for_function(
                """() => {
                    const div = document.querySelector('#caseHistoryDiv');
                    if (!div) return false;
                    const text = div.innerText.toLowerCase();
                    return text.includes('invalid captcha') || text.includes('cnr does not exits');
                }""",
                timeout=timeout,
                polling=250
            )
            return 'invalid'
        except Exception:
            return 'timeout_invalid'

    tasks = [
        asyncio.create_task(wait_valid()),
        asyncio.create_task(wait_invalid_texts())
    ]

    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

    for task in pending:
        task.cancel()  # Stop unused ones

    result = await list(done)[0]

    return result == 'valid'


In [25]:

from time import time
import gc
# Needed to allow nested event loops in Jupyter
nest_asyncio.apply()

async def run():
    case_info.clear()
    # Create a folder with the current date and time in the name
    # current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    # folder_name = f"data_link_{current_time}"
    # os.makedirs(folder_name, exist_ok=True)
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context(viewport={"width": 1280, "height": 720})
        page = await context.new_page()
        await page.goto("https://hcservices.ecourts.gov.in/hcservices/main.php")
        i = 0
        while i < len(cnr_numbers):
            start_time = time()
            cnr_number = cnr_numbers[i]
            print(cnr_number)
            img = await get_captcha(page)
            time1 = time()
            print("time1:",time1-start_time)
            # await denoise_image2('captcha.png')
            cap_text = await ML_captcha_solver4(img)
            time2 = time()
            print("time2:",time2-time1)
            # cap_text = ''.join(['0' if not ch.isalnum() else ch for ch in cap_text])
            # if(len(cap_text) < 6):
            #     cap_text = '000000'
            await page.fill('input[placeholder="Enter CNR number"]', cnr_number)
            await page.fill('input[id="captcha"]', cap_text)
            await page.click('[id="searchbtn"]')
            time3 = time()
            print("time3:",time3-time2)
            err = await page.query_selector('div#errSpan')
            err_text = await err.inner_text() if err else None
            if err_text:
                print(err_text)
                i += 1
                continue
            valid_captcha = await is_captcha_valid(page)
            time4 = time()
            print("time4:",time4-time3)
            if not valid_captcha:
                print("Invalid Captcha")
            else:
                valid_cnr = True
                div = await page.query_selector('#caseHistoryDiv')
                if div:
                    text = await div.inner_text()
                    if 'cnr does not exits' in text.lower():
                        print("CNR does not exist")
                        valid_cnr = False
                if valid_cnr:
                    await get_links(cnr_number, page, context)
                time5 = time()
                print("time5:",time5-time4)
                i += 1
            # await page.click('input[id="bckbtn"]')
            await page.goto("https://hcservices.ecourts.gov.in/hcservices/main.php")
            end_time = time()
            print("time_taken: ", end_time - start_time)
            # if i % 2000 == 0:
            #     output_file = os.path.join(f"data_link_{court.lower()}", f"case_info_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
            #     with open(output_file, "w") as f:
            #         json.dump(case_info, f, indent=4)
            #     case_info.clear()
            #     gc.collect()
            # # await page.wait_for_timeout(2000)
            print("-------------")
        print(f"Page title: {await page.title()}")
        await browser.close()
        
    # Dump case_info into a JSON file
    # output_file = os.path.join(f"data_link_{court.lower()}", f"case_info_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    # with open(output_file, "w") as f:
    #     json.dump(case_info, f, indent=4)

In [22]:
# current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
# output_file = os.path.join(f'data_link_{court.lower()}', f"case_info_{current_time}.json")
# with open(output_file, "w") as f:
#     json.dump(case_info, f, indent=4)

In [ ]:
await run()

ODHC010384572015
time1: 0.053351640701293945
time2: 0.0702826976776123


/home/uddeshya-raj/ibps/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


time3: 0.11356687545776367
time4: 1.539637565612793
CNR: ODHC010384572015
Link: cases/display_pdf.php?filename=oVz058q2ZLjDVEITM0Vw1AJ5svTq4hn6AfmVcuSR8VW%2B32OpxcxWEiUmCUYNXcNf&caseno=ABLAPL/19196/2015&cCode=1&cino=ODHC010384572015&state_code=11&court_code=1&&appFlag=
https://hcservices.ecourts.gov.in/hcservices/cases/display_pdf.php?filename=oVz058q2ZLjDVEITM0Vw1AJ5svTq4hn6AfmVcuSR8VW%2B32OpxcxWEiUmCUYNXcNf&caseno=ABLAPL/19196/2015&cCode=1&cino=ODHC010384572015&state_code=11&court_code=1&&appFlag=
Link: cases/display_pdf.php?filename=oVz058q2ZLjDVEITM0Vw1AJ5svTq4hn6AfmVcuSR8VWrs9LxT0uGv3fLKZ8hw8Ed&caseno=ABLAPL/19196/2015&cCode=1&cino=ODHC010384572015&state_code=11&court_code=1&&appFlag=
https://hcservices.ecourts.gov.in/hcservices/cases/display_pdf.php?filename=oVz058q2ZLjDVEITM0Vw1AJ5svTq4hn6AfmVcuSR8VWrs9LxT0uGv3fLKZ8hw8Ed&caseno=ABLAPL/19196/2015&cCode=1&cino=ODHC010384572015&state_code=11&court_code=1&&appFlag=
Link: cases/display_pdf.php?filename=oVz058q2ZLjDVEITM0Vw1AJ5svTq4hn

In [ ]:
# bail_data = pd.read_csv(r'Bail - Case dataset/Bail Bombay.csv')
# cnr_numbers = bail_data['CNR_NUMBER'].tolist()

: 

: 

In [ ]:
# await run()

: 

: 